# ODIR-5K — Шаг 5: Сравнение моделей машинного обучения
**Дипломная работа:** Разработка и исследование моделей МО для диагностики офтальмологических заболеваний

### Сравниваем 4 архитектуры:
| Модель | Параметры | Особенность |
|--------|-----------|-------------|
| **EfficientNet-B0** | 5.3M | Наш baseline, лучшее соотношение точность/размер |
| **ResNet50** | 25.6M | Классика, остаточные связи |
| **DenseNet121** | 8M | Плотные связи, популярна в медицине |
| **MobileNetV3** | 5.4M | Лёгкая, быстрая, для мобильных устройств |

## Шаг 1 — Подготовка окружения

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q albumentations

import json, time, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import roc_auc_score, f1_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DRIVE_DIR = Path('/content/drive/MyDrive/diploma_odir')
COMPARE_DIR = DRIVE_DIR / 'comparison'
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('✅ Готово')

## Шаг 2 — Конфигурация и данные

In [ ]:
# Скачиваем датасет если нужно
if not Path('/content/data').exists():
    print('Скачиваем датасет...')
    !kaggle datasets download -d andrewmvd/ocular-disease-recognition-odir5k \
        -p /content/data --unzip -q

train_dirs = list(Path('/content/data').rglob('Training Images'))
TRAIN_IMG_DIR = train_dirs[0]

CLASSES = ['N','D','G','C','A','H','M','O']
CLASS_NAMES = {'N':'Normal','D':'Diabetes','G':'Glaucoma','C':'Cataract',
               'A':'AMD','H':'Hypertension','M':'Myopia','O':'Other'}

# Гиперпараметры — одинаковые для всех моделей (честное сравнение!)
IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_EPOCHS  = 15      # достаточно для сравнения
LR          = 1e-4
RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f'IMG_SIZE={IMG_SIZE} | BATCH={BATCH_SIZE} | EPOCHS={NUM_EPOCHS} | LR={LR}')

In [ ]:
from sklearn.model_selection import train_test_split

# Загрузка данных
csv_files = list(Path('/content/data').rglob('full_df.csv'))
df_raw = pd.read_csv(csv_files[0])
df_raw.columns = [c.strip() for c in df_raw.columns]
df_raw.rename(columns={'Left-Fundus':'filename_left','Right-Fundus':'filename_right'}, inplace=True)
for cls in CLASSES:
    if cls in df_raw.columns:
        df_raw[cls] = df_raw[cls].astype(int)

# Разворачиваем пары
rows = []
for _, row in df_raw.iterrows():
    for side in ('left','right'):
        fname = row[f'filename_{side}']
        fpath = TRAIN_IMG_DIR / fname
        rows.append({'filepath':str(fpath),'filename':fname,'side':side,
                     **{cls:row[cls] for cls in CLASSES if cls in row}})

df = pd.DataFrame(rows)
df = df[df['filepath'].apply(lambda p: Path(p).exists())].reset_index(drop=True)

# Разбивка 70/15/15
df['_strat'] = df[CLASSES].apply(
    lambda r: int(np.argmax(r.values)) if r.values.sum()>0 else -1, axis=1)
train_val, test_df = train_test_split(df, test_size=0.15, random_state=RANDOM_SEED, stratify=df['_strat'])
train_df, val_df   = train_test_split(train_val, test_size=0.15/0.85, random_state=RANDOM_SEED, stratify=train_val['_strat'])
train_df = train_df.drop(columns=['_strat']).reset_index(drop=True)
val_df   = val_df.drop(columns=['_strat']).reset_index(drop=True)
test_df  = test_df.drop(columns=['_strat']).reset_index(drop=True)

# Веса классов
pos_counts = train_df[CLASSES].sum(axis=0).values
pos_weight = torch.tensor((len(train_df)-pos_counts)/(pos_counts+1e-8), dtype=torch.float32)

print(f'Train:{len(train_df)} | Val:{len(val_df)} | Test:{len(test_df)}')
print('✅ Данные готовы')

In [ ]:
# Dataset и трансформации
def get_train_transforms():
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.Rotate(limit=30, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
        A.GaussNoise(var_limit=(10,50), p=0.2),
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2(),
    ])

def get_val_transforms():
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2(),
    ])

class ODIRDataset(Dataset):
    def __init__(self, df, transforms):
        self.df=df.reset_index(drop=True); self.transforms=transforms
        self.classes=[c for c in CLASSES if c in df.columns]
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        try: image=np.array(Image.open(row['filepath']).convert('RGB'))
        except: image=np.zeros((IMG_SIZE,IMG_SIZE,3),dtype=np.uint8)
        image_t=self.transforms(image=image)['image']
        labels=torch.tensor(row[self.classes].values.astype(np.float32))
        return image_t, labels

train_loader = DataLoader(ODIRDataset(train_df,get_train_transforms()), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(ODIRDataset(val_df,  get_val_transforms()),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(ODIRDataset(test_df, get_val_transforms()),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print('✅ DataLoader готов')

## Шаг 3 — Определение моделей

In [ ]:
def build_efficientnet_b0(num_classes=8):
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    m.classifier = nn.Sequential(nn.Dropout(p=0.3,inplace=True), nn.Linear(m.classifier[1].in_features, num_classes))
    return m

def build_resnet50(num_classes=8):
    m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    m.fc = nn.Sequential(nn.Dropout(p=0.3), nn.Linear(m.fc.in_features, num_classes))
    return m

def build_densenet121(num_classes=8):
    m = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
    m.classifier = nn.Sequential(nn.Dropout(p=0.3), nn.Linear(m.classifier.in_features, num_classes))
    return m

def build_mobilenet_v3(num_classes=8):
    m = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
    m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes)
    return m

# Словарь всех моделей
MODEL_BUILDERS = {
    'EfficientNet-B0': build_efficientnet_b0,
    'ResNet50':        build_resnet50,
    'DenseNet121':     build_densenet121,
    'MobileNetV3':     build_mobilenet_v3,
}

# Проверим количество параметров
print(f'{'Модель':<20} {'Параметров':>15}')
print('-'*37)
for name, builder in MODEL_BUILDERS.items():
    m = builder()
    params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'{name:<20} {params:>15,}')
    del m

## Шаг 4 — Функции обучения и оценки

In [ ]:
def compute_auc(labels, probs):
    aucs = []
    for i in range(len(CLASSES)):
        if labels[:,i].sum() > 0:
            aucs.append(roc_auc_score(labels[:,i], probs[:,i]))
    return float(np.mean(aucs)) if aucs else 0.0

def compute_f1(labels, probs, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    return float(f1_score(labels, preds, average='macro', zero_division=0))

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, all_labels, all_probs = 0.0, [], []
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        with torch.no_grad():
            all_probs.append(torch.sigmoid(model(images)).cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    all_probs  = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)
    return total_loss/len(loader.dataset), compute_auc(all_labels,all_probs), compute_f1(all_labels,all_probs)

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, all_labels, all_probs = 0.0, [], []
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        logits = model(images)
        total_loss += criterion(logits, labels).item() * images.size(0)
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    all_probs  = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)
    return total_loss/len(loader.dataset), compute_auc(all_labels,all_probs), compute_f1(all_labels,all_probs), all_probs, all_labels

def train_model(name, builder):
    print(f'\n{'='*55}')
    print(f'  Обучение: {name}')
    print(f'{'='*55}')

    model = builder(len(CLASSES)).to(DEVICE)
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Параметров: {params:,}')

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

    history = {'train_loss':[],'val_loss':[],'train_auc':[],'val_auc':[],'train_f1':[],'val_f1':[]}
    best_val_auc, no_improve = 0.0, 0
    best_path = COMPARE_DIR / f'best_{name.replace("-","_").lower()}.pth'
    total_time = 0

    for epoch in range(1, NUM_EPOCHS+1):
        t0 = time.time()
        tr_loss, tr_auc, tr_f1 = train_one_epoch(model, train_loader, optimizer, criterion)
        vl_loss, vl_auc, vl_f1, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step(vl_loss)
        elapsed = time.time()-t0
        total_time += elapsed

        history['train_loss'].append(tr_loss); history['val_loss'].append(vl_loss)
        history['train_auc'].append(tr_auc);   history['val_auc'].append(vl_auc)
        history['train_f1'].append(tr_f1);     history['val_f1'].append(vl_f1)

        print(f'  Epoch {epoch:>2}/{NUM_EPOCHS} | loss={tr_loss:.4f}/{vl_loss:.4f} | auc={tr_auc:.4f}/{vl_auc:.4f} | f1={tr_f1:.4f}/{vl_f1:.4f} | {elapsed:.0f}s')

        if vl_auc > best_val_auc:
            best_val_auc = vl_auc
            no_improve = 0
            torch.save({'model_state':model.state_dict(),'val_auc':vl_auc,'epoch':epoch,'name':name}, best_path)
            print(f'  ✅ Best AUC: {best_val_auc:.4f}')
        else:
            no_improve += 1
            if no_improve >= 5:
                print(f'  ⏹ Early stopping на эпохе {epoch}')
                break

    # Финальная оценка на TEST
    ckpt = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    _, test_auc, test_f1, test_probs, test_labels = evaluate(model, test_loader, criterion)

    # Per-class AUC
    per_class_auc = {}
    for i, cls in enumerate(CLASSES):
        if test_labels[:,i].sum() > 0:
            per_class_auc[cls] = round(float(roc_auc_score(test_labels[:,i], test_probs[:,i])), 4)

    result = {
        'name':          name,
        'params':        params,
        'best_val_auc':  round(best_val_auc, 4),
        'test_auc':      round(test_auc, 4),
        'test_f1':       round(test_f1, 4),
        'total_time_min': round(total_time/60, 1),
        'per_class_auc': per_class_auc,
        'history':       history,
    }

    print(f'\n  TEST AUC={test_auc:.4f} | F1={test_f1:.4f} | Время={total_time/60:.1f} мин')
    return result

print('✅ Функции готовы')

## Шаг 5 — Обучение всех моделей
⏱ Ожидаемое время: ~40-60 минут на T4 GPU

In [ ]:
all_results = {}
total_start = time.time()

for model_name, builder in MODEL_BUILDERS.items():
    result = train_model(model_name, builder)
    all_results[model_name] = result

    # Сохраняем результат каждой модели сразу
    with open(COMPARE_DIR / f'result_{model_name.replace("-","_").lower()}.json', 'w') as f:
        json.dump({k:v for k,v in result.items() if k != 'history'}, f, indent=2)

print(f'\n✅ Все модели обучены! Общее время: {(time.time()-total_start)/60:.0f} мин')

## Шаг 6 — Сравнительная таблица

In [ ]:
# Таблица результатов
rows = []
for name, r in all_results.items():
    rows.append({
        'Модель':       name,
        'Параметров':   f"{r['params']:,}",
        'Val AUC':      r['best_val_auc'],
        'Test AUC':     r['test_auc'],
        'Test F1':      r['test_f1'],
        'Время (мин)':  r['total_time_min'],
    })

df_results = pd.DataFrame(rows).sort_values('Test AUC', ascending=False).reset_index(drop=True)
df_results.index += 1

print('\n' + '='*70)
print('  СРАВНИТЕЛЬНАЯ ТАБЛИЦА МОДЕЛЕЙ — ODIR-5K (TEST SET)')
print('='*70)
print(df_results.to_string())
print('='*70)
print(f"\n🏆 Лучшая модель: {df_results.iloc[0]['Модель']} (AUC={df_results.iloc[0]['Test AUC']})")

df_results.to_csv(COMPARE_DIR / 'comparison_table.csv', index=False)
print('✅ Таблица сохранена')

## Шаг 7 — Графики сравнения

In [ ]:
colors_map = {
    'EfficientNet-B0': '#1a237e',
    'ResNet50':        '#c62828',
    'DenseNet121':     '#2e7d32',
    'MobileNetV3':     '#f57f17',
}

fig = plt.figure(figsize=(18, 14))
fig.suptitle('Сравнение моделей машинного обучения — ODIR-5K', fontsize=16, fontweight='bold', y=0.98)

# 1. Val AUC по эпохам
ax1 = fig.add_subplot(3, 3, 1)
for name, r in all_results.items():
    epochs = range(1, len(r['history']['val_auc'])+1)
    ax1.plot(epochs, r['history']['val_auc'], color=colors_map[name], label=name, linewidth=2)
ax1.set_title('Val AUC по эпохам', fontweight='bold')
ax1.set_xlabel('Эпоха'); ax1.set_ylabel('AUC-ROC')
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)
ax1.set_ylim(0.5, 1.0)

# 2. Val Loss по эпохам
ax2 = fig.add_subplot(3, 3, 2)
for name, r in all_results.items():
    epochs = range(1, len(r['history']['val_loss'])+1)
    ax2.plot(epochs, r['history']['val_loss'], color=colors_map[name], label=name, linewidth=2)
ax2.set_title('Val Loss по эпохам', fontweight='bold')
ax2.set_xlabel('Эпоха'); ax2.set_ylabel('Loss')
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

# 3. Val F1 по эпохам
ax3 = fig.add_subplot(3, 3, 3)
for name, r in all_results.items():
    epochs = range(1, len(r['history']['val_f1'])+1)
    ax3.plot(epochs, r['history']['val_f1'], color=colors_map[name], label=name, linewidth=2)
ax3.set_title('Val F1 по эпохам', fontweight='bold')
ax3.set_xlabel('Эпоха'); ax3.set_ylabel('F1 macro')
ax3.legend(fontsize=8); ax3.grid(alpha=0.3)

# 4. Test AUC (bar)
ax4 = fig.add_subplot(3, 3, 4)
names = [r['name'] for r in all_results.values()]
aucs  = [r['test_auc'] for r in all_results.values()]
bars = ax4.bar(names, aucs, color=[colors_map[n] for n in names], alpha=0.85, width=0.5)
ax4.set_title('Test AUC-ROC', fontweight='bold')
ax4.set_ylabel('AUC-ROC'); ax4.set_ylim(0.5, 1.0)
ax4.tick_params(axis='x', rotation=15)
for bar, val in zip(bars, aucs):
    ax4.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax4.grid(axis='y', alpha=0.3)

# 5. Test F1 (bar)
ax5 = fig.add_subplot(3, 3, 5)
f1s = [r['test_f1'] for r in all_results.values()]
bars2 = ax5.bar(names, f1s, color=[colors_map[n] for n in names], alpha=0.85, width=0.5)
ax5.set_title('Test F1 macro', fontweight='bold')
ax5.set_ylabel('F1'); ax5.set_ylim(0, 1.0)
ax5.tick_params(axis='x', rotation=15)
for bar, val in zip(bars2, f1s):
    ax5.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax5.grid(axis='y', alpha=0.3)

# 6. Время обучения (bar)
ax6 = fig.add_subplot(3, 3, 6)
times = [r['total_time_min'] for r in all_results.values()]
bars3 = ax6.bar(names, times, color=[colors_map[n] for n in names], alpha=0.85, width=0.5)
ax6.set_title('Время обучения (мин)', fontweight='bold')
ax6.set_ylabel('Минуты')
ax6.tick_params(axis='x', rotation=15)
for bar, val in zip(bars3, times):
    ax6.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2, f'{val:.1f}', ha='center', va='bottom', fontsize=9)
ax6.grid(axis='y', alpha=0.3)

# 7. AUC по классам (grouped bar)
ax7 = fig.add_subplot(3, 1, 3)
x = np.arange(len(CLASSES))
width = 0.2
for i, (name, r) in enumerate(all_results.items()):
    vals = [r['per_class_auc'].get(cls, 0) for cls in CLASSES]
    ax7.bar(x + i*width - 0.3, vals, width, label=name, color=colors_map[name], alpha=0.85)
ax7.set_title('AUC-ROC по классам заболеваний', fontweight='bold')
ax7.set_xticks(x); ax7.set_xticklabels([CLASS_NAMES[c] for c in CLASSES], rotation=15, ha='right')
ax7.set_ylabel('AUC-ROC'); ax7.set_ylim(0.4, 1.05)
ax7.legend(); ax7.grid(axis='y', alpha=0.3)
ax7.axhline(y=0.8, color='gray', linestyle='--', linewidth=0.8, alpha=0.6, label='Порог 0.8')

plt.tight_layout()
out = COMPARE_DIR / 'model_comparison.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ График сохранён → {out}')

## Шаг 8 — Итоговый вывод для диплома

In [ ]:
best = df_results.iloc[0]
worst = df_results.iloc[-1]

print('='*65)
print('  ВЫВОДЫ ДЛЯ ДИПЛОМНОЙ РАБОТЫ')
print('='*65)
print(f"""
В ходе исследования были обучены и сравнены 4 архитектуры
нейронных сетей на датасете ODIR-5K (офтальмологические
заболевания, 8 классов, ~10 000 снимков).

Все модели обучались в одинаковых условиях:
  - Оптимизатор: AdamW (lr={LR}, weight_decay=1e-4)
  - Loss: BCEWithLogitsLoss с pos_weight (борьба с дисбалансом)
  - Эпох: до {NUM_EPOCHS} (с early stopping)
  - Размер батча: {BATCH_SIZE}
  - Аугментация: flip, rotate, brightness, noise

РЕЗУЛЬТАТЫ (по Test AUC-ROC):
""")

for i, row in df_results.iterrows():
    medal = '🥇' if i==1 else '🥈' if i==2 else '🥉' if i==3 else '  '
    print(f'  {medal} {i}. {row["Модель"]:<18} AUC={row["Test AUC"]}  F1={row["Test F1"]}  Время={row["Время (мин)"]} мин')

print(f"""
ВЫВОД:
  Лучшей моделью является {best['Модель']} с AUC-ROC = {best['Test AUC']}.
  Наименее эффективной — {worst['Модель']} (AUC = {worst['Test AUC']}).

  {best['Модель']} выбрана в качестве основной модели системы
  благодаря наилучшему соотношению точности и вычислительной
  эффективности, что критически важно для медицинских приложений.
""")
print('='*65)

## Шаг 9 — Сохранение всего в Google Drive

In [ ]:
# Сохраняем полные результаты
full_results = {name: {k:v for k,v in r.items()} for name,r in all_results.items()}
with open(COMPARE_DIR / 'all_results.json', 'w') as f:
    json.dump(full_results, f, indent=2)

print('✅ Все результаты сохранены в Drive:')
print('📁 MyDrive/diploma_odir/comparison/')
for f in sorted(COMPARE_DIR.iterdir()):
    size = f.stat().st_size / 1e3
    print(f'  {f.name:<45} ({size:.0f} KB)')